# 09 · Own pipeline, final version

The pipeline of notebook 07 (flip TTA → ensemble → temporal rescoring) rebuilt around the stronger RT-DETR
models of notebook 08, with two changes that follow from the comparison of val and test:

- **Decisions on test-like val.** All test frames are at most 1920 px wide, 55% of the val frames are
  2.7K–4K. Every setting here is chosen on the val frames at most 1920 px wide (1,286 frames, 4 clips), the
  TTA settings also on the same frames shrunk to 960 px (16% of the test frames are 680–960 px wide). The
  public leaderboard covers only 30% of the test set and the grade uses the other 70%, so it is not used to
  choose anything.
- **Scale TTA.** Besides the mirrored frame, the frame enlarged 1.33× (1440 × 2560 input) is tried: higher
  input resolution was the largest single gain in notebook 08, and test-time enlargement costs no training.

Steps: members and their test-like scores → TTA search (GPU, every 5th test-like val frame) → chosen TTA on
the test-like val frames and the test set → ensemble search → temporal rescoring → final submission.

Runs on Kaggle with two inputs: the output of the last notebook 08 session (all RT-DETR runs with
checkpoints) and the `yolo-p2-run` dataset (notebook 06).

## Setup

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-object-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "albumentations"],
                   capture_output=True)   # Ultralytics would apply it automatically; not used here
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwdet import env
from hwdet.data.download import ensure_data
from hwdet.tracker import Run, status
P = env.setup(seed=42)
ensure_data(P, drive_cache=False)   # Colab: the zip goes to the local disk, not Drive
status(P)                           # which runs / stages are already done

In [ ]:
# Imports, plot style
import itertools
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

from hwdet.config import load_config
from hwdet.data.detection_dataset import DetectionDataset, build_targets
from hwdet.data.io import frame_table
from hwdet.engine import load_weights, predict
from hwdet.env import REPO_ROOT
from hwdet.eval.coco_eval import evaluate, evaluate_by_size
from hwdet.pipeline import DownscaledDetector, MultiScaleDetector, merge_predictions, temporal_rescore
from hwdet.rtdetr import build_rtdetr
from hwdet.submission import make_submission
from hwdet.viz import INK_3, PALETTE, save_fig, set_style

NOTEBOOK_START = time.time()
set_style()
dev = env.device()
print("torch", torch.__version__, "| device", dev)

## 1. Data

In [ ]:
# Val frames at most 1920 px wide (test-like), every 5th of them for the GPU search, the test set
val_frames = frame_table(P.data, "val")
test_frames = frame_table(P.data, "test")
targets = build_targets(P.data, val_frames)
like_test = val_frames[val_frames.width <= 1920]
search_frames = like_test.iloc[::5]
LOW_SHARE = 0.16                          # share of test frames 680-960 px wide
print(f"{len(like_test)} test-like val frames ({like_test.clip.nunique()} clips), {len(search_frames)} for the TTA search,"
      f" {len(test_frames)} test frames")

## 2. Members

In [ ]:
# Runs of notebooks 06 and 08 with saved val and test predictions; test-like scores saved by notebook 08
CANDIDATES = ["rtdetr_r101", "rtdetr_v2", "rtdetr_lowres", "rtdetr_hires", "rtdetr_data3", "yolo_p2"]
MEMBERS = [r for r in CANDIDATES if (P.runs / r / "predictions" / "test.csv").exists()]
missing = [r for r in ("rtdetr_hires", "yolo_p2") if r not in MEMBERS]
assert not missing, f"runs {missing} missing: attach the last notebook 08 output and the yolo-p2-run dataset"
pv = {r: pd.read_csv(P.runs / r / "predictions" / "val.csv") for r in MEMBERS}
pt = {r: pd.read_csv(P.runs / r / "predictions" / "test.csv") for r in MEMBERS}
ms = {r: json.loads((P.runs / r / "metrics.json").read_text())["val"]["ms_per_img"] for r in MEMBERS}


def like(preds, frames=like_test):
    """Main COCO numbers on test-like val frames."""
    m = evaluate(preds[preds.file_name.isin(frames.file_name)], frames, P.data, per_class=False)
    return {k: m[k] for k in ("mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR_500")}


rows = {}
for r in MEMBERS:
    tl = P.runs / r / "test_like.json"
    rows[r] = {**like(pv[r]), "test-like mAP (notebook 08)": json.loads(tl.read_text())["test-like mAP"] if tl.exists() else np.nan,
               "full val mAP": json.loads((P.runs / r / "metrics.json").read_text())["val"]["mAP"], "ms_per_img": ms[r]}
base = pd.DataFrame(rows).T
RTDETRS = [r for r in base.index if r.startswith("rtdetr")]
TTA_MODELS = base.loc[RTDETRS, "test-like mAP (notebook 08)"].fillna(base.loc[RTDETRS, "mAP"]).nlargest(2).index.tolist()
print("TTA for:", TTA_MODELS)
base.round(4)

## 3. Test-time augmentation

Settings: none, the mirrored frame (flip), the frame enlarged 1.33×, and both together (4 passes). Boxes of
all passes are merged with class-wise NMS at IoU 0.5. Each setting is scored on every 5th test-like val
frame as it is and shrunk to 960 px wide: TTA score = 0.84 · mAP + 0.16 · mAP at 960 px. The best setting
whose run on the test-like val frames and the test set fits the time budget is kept.

In [ ]:
# Search: every setting x the two best RT-DETR runs; rows are saved as they finish (resumable)
SETTINGS = {"none": {"scales": [1.0]}, "flip": {"scales": [1.0], "flip": True},
            "1.33x": {"scales": [1.33]}, "flip + 1.33x": {"scales": [1.0, 1.33], "flip": True}}
models = {r: load_weights(build_rtdetr(load_config(r)), Run(r, None, P), "best", dev) for r in TTA_MODELS}
search_run = Run("pipe2_tta_search", {"family": "pipeline", "description": "TTA search, every 5th test-like val frame"}, P)
f = search_run.dir / "search.csv"
rows = pd.read_csv(f).to_dict("records") if f.exists() else []
done = {(r["model"], r["setting"]) for r in rows}
search_ds = DetectionDataset(search_frames, P.data, targets)
for name, model in models.items():
    for s, kw in SETTINGS.items():
        if (name, s) in done:
            continue
        det = MultiScaleDetector(model, **kw)
        t0 = time.time()
        p = predict(det, search_ds, dev, batch_size=1, workers=2, desc=f"{name} {s}")
        sec = (time.time() - t0) / len(search_frames)
        low = predict(DownscaledDetector(det, 960), search_ds, dev, batch_size=1, workers=2, desc=f"{name} {s} 960 px")
        m, m_low = like(p, search_frames)["mAP"], like(low, search_frames)["mAP"]
        rows.append({"model": name, "setting": s, "mAP": m, "mAP at 960 px": m_low,
                     "TTA score": (1 - LOW_SHARE) * m + LOW_SHARE * m_low, "s_per_img": sec})
        pd.DataFrame(rows).to_csv(f, index=False)
search = pd.DataFrame(rows)
print(f"elapsed: {(time.time() - NOTEBOOK_START) / 3600:.2f} h")
search.round(4)

In [ ]:
# Best setting per model by TTA score, among settings whose test-like val + test run fits the time budget
BUDGET_H = 3.0                            # per model; notebook 08 leaves ~18 of the 30 weekly GPU hours
n_full = len(like_test) + len(test_frames)
search["hours_full_run"] = search.s_per_img * n_full / 3600
chosen = {}
for name in TTA_MODELS:
    s = search[(search.model == name) & (search.hours_full_run <= BUDGET_H)]
    chosen[name] = s.loc[s["TTA score"].idxmax(), "setting"]
print("chosen:", chosen)
search.pivot(index="setting", columns="model", values=["TTA score", "mAP", "mAP at 960 px", "hours_full_run"]).round(3)

In [ ]:
# Chosen TTA on the test-like val frames (metrics + speed) and on the test set; one run per model
tta_v, tta_t = dict(pv), dict(pt)
for name, setting in chosen.items():
    if setting == "none":
        continue
    rid = f"pipe2_{name.removeprefix('rtdetr_')}_tta"
    run = Run(rid, {"family": "pipeline", "model": {"arch": name}, "tta": {"setting": setting, **SETTINGS[setting]},
                    "description": f"{name} with TTA: {setting}, NMS 0.5"}, P)
    det = MultiScaleDetector(models[name], **SETTINGS[setting])
    if not run.is_done("eval_val"):
        p = predict(det, DetectionDataset(like_test, P.data, targets), dev, batch_size=1, workers=2, desc=f"{rid} val")
        p.to_csv(run.pred_path("val"), index=False)
        sec = search[(search.model == name) & (search.setting == setting)].s_per_img.iloc[0]
        run.log_metrics("val_like_test", {**evaluate(p, like_test, P.data), "ms_per_img": 1000 * sec})
        run.mark_done("eval_val")
    if not run.is_done("test"):
        p = predict(det, DetectionDataset(test_frames, P.data), dev, batch_size=1, workers=2, desc=f"{rid} test")
        p.to_csv(run.pred_path("test"), index=False)
        make_submission(p, test_frames, run.dir / "submission.csv")
        run.mark_done("test")
    tta_v[name], tta_t[name] = pd.read_csv(run.pred_path("val")), pd.read_csv(run.pred_path("test"))
    ms[name] = run.metrics["val_like_test"]["ms_per_img"]
print(f"elapsed: {(time.time() - NOTEBOOK_START) / 3600:.2f} h")
pd.DataFrame({n: {**like(tta_v[n]), "TTA": chosen.get(n, "none"), "ms_per_img": ms[n]} for n in MEMBERS}).T

## 4. Ensemble

The best member plus every combination of the others, merged with class-wise NMS at IoU 0.6 (the better
value in notebook 07); the other members' scores at full or half weight. Scored on all test-like val frames.

In [ ]:
# Ensemble search on the test-like val frames
member_v = {n: tta_v[n] for n in MEMBERS}
lead_scores = {n: like(member_v[n])["mAP"] for n in MEMBERS}
LEAD = max(lead_scores, key=lead_scores.get)
label = {n: f"{n} ({chosen.get(n, 'none')})" for n in MEMBERS}
others = [m for m in MEMBERS if m != LEAD]
rows = [{"members": label[LEAD], "combo": (LEAD,), "weight_others": np.nan, **like(member_v[LEAD])}]
for k in range(1, len(others) + 1):
    for extra in itertools.combinations(others, k):
        for w in (1.0, 0.5):
            p = merge_predictions([member_v[c] for c in (LEAD,) + extra], iou=0.6, weights=[1.0] + [w] * k)
            rows.append({"members": " + ".join(label[c] for c in (LEAD,) + extra), "combo": (LEAD,) + extra,
                         "weight_others": w, **like(p)})
ens = pd.DataFrame(rows).sort_values("mAP", ascending=False).reset_index(drop=True)
print(f"elapsed: {(time.time() - NOTEBOOK_START) / 3600:.2f} h")
ens.drop(columns="combo").round(4)

In [ ]:
# Best ensemble on the test-like val frames and the test set
best = ens.iloc[0]
if len(best.combo) == 1:
    ens_v, ens_t = tta_v[LEAD], tta_t[LEAD]
else:
    weights = [1.0] + [best.weight_others] * (len(best.combo) - 1)
    ens_v = merge_predictions([tta_v[c] for c in best.combo], iou=0.6, weights=weights)
    ens_t = merge_predictions([tta_t[c] for c in best.combo], iou=0.6, weights=weights)
ens_v = ens_v[ens_v.file_name.isin(like_test.file_name)]
print("ensemble:", best.members, "| weight of the others:", best.weight_others)
pd.Series(like(ens_v)).round(4)

## 5. Temporal rescoring

As in notebook 07: each box's score is mixed with the score of its best same-class match (IoU ≥ 0.5) in
the `window` previous and next frames; λ is the weight of the neighbours. Smaller λ values are tried than
before, as λ = 0.25 was the best and 0.75 hurt.

In [ ]:
# Search window x lambda on the test-like val frames
rows = [{"window": 0, "lam": 0.0, **like(ens_v)}]
for window in (1, 2):
    for lam in (0.15, 0.25, 0.4):
        rows.append({"window": window, "lam": lam, **like(temporal_rescore(ens_v, like_test, window=window, iou_thr=0.5, lam=lam))})
temporal = pd.DataFrame(rows).sort_values("mAP", ascending=False).reset_index(drop=True)
temporal.round(4)

## 6. Final pipeline and submissions

In [ ]:
# Final pipeline = best ensemble + best temporal setting; a second submission without the ensemble as a fallback
w, lam = int(temporal.window[0]), float(temporal.lam[0])
final_v = temporal_rescore(ens_v, like_test, w, 0.5, lam) if w else ens_v
final_t = temporal_rescore(ens_t, test_frames, w, 0.5, lam) if w else ens_t
final = Run("pipe2_final", {"family": "pipeline", "model": {"arch": "TTA + ensemble + temporal"},
                            "description": f"{best.members}, NMS IoU 0.6, weight of the others {best.weight_others}, "
                                           f"temporal window {w}, lambda {lam}"}, P)
final_v.to_csv(final.pred_path("val"), index=False)
final_t.to_csv(final.pred_path("test"), index=False)
final.log_metrics("val_like_test", {**evaluate(final_v, like_test, P.data), "ms_per_img": sum(ms[c] for c in best.combo)})
make_submission(final_t, test_frames, final.dir / "submission.csv")

lead_v = tta_v[LEAD][tta_v[LEAD].file_name.isin(like_test.file_name)]
single_v = temporal_rescore(lead_v, like_test, w, 0.5, lam) if w else lead_v
single_t = temporal_rescore(tta_t[LEAD], test_frames, w, 0.5, lam) if w else tta_t[LEAD]
single = Run("pipe2_single", {"family": "pipeline", "model": {"arch": "TTA + temporal"},
                              "description": f"{label[LEAD]} + temporal window {w}, lambda {lam} (no ensemble)"}, P)
single_v.to_csv(single.pred_path("val"), index=False)
single_t.to_csv(single.pred_path("test"), index=False)
single.log_metrics("val_like_test", {**evaluate(single_v, like_test, P.data), "ms_per_img": ms[LEAD]})
make_submission(single_t, test_frames, single.dir / "submission.csv")
pd.DataFrame({"pipe2_final": final.metrics["val_like_test"], "pipe2_single": single.metrics["val_like_test"]}).T[
    ["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR_500", "ms_per_img"]].round(4)

## 7. What each step adds

On the test-like val frames, from the best member as trained to the final pipeline. The settings were
chosen on these frames, so the last rows are slightly optimistic.

In [ ]:
# Ablation: best member -> + TTA -> + ensemble -> + temporal
steps = {f"{LEAD} (as trained)": pv[LEAD], f"+ TTA ({chosen.get(LEAD, 'none')})": tta_v[LEAD],
         f"+ ensemble ({best.members})": ens_v, f"+ temporal (window {w}, λ {lam})": final_v}
pd.DataFrame({k: like(v) for k, v in steps.items()}).T.round(4)

In [ ]:
# AP by object size (sides in original pixels) on the test-like val frames: best member vs final pipeline
pd.concat({"best member": evaluate_by_size(pv[LEAD][pv[LEAD].file_name.isin(like_test.file_name)], like_test, P.data),
           "final pipeline": evaluate_by_size(final_v, like_test, P.data)}, names=["model", "object side"]).round(4)

## 8. Summary of all models

In [ ]:
# Every model of the homework: full val mAP (results/experiments.csv) and mAP on the test-like val frames
old = pd.read_csv(REPO_ROOT / "results" / "experiments.csv")
summary = old[old["eval"] == "val"].set_index("run_id")[["family", "mAP", "AP_small", "ms_per_img"]].rename(columns={"mAP": "full val mAP"})
for d in sorted(P.runs.iterdir()):
    if (d / "predictions" / "val.csv").exists() and (d / "metrics.json").exists():
        c = (yaml.safe_load((d / "config.yaml").read_text()) or {}) if (d / "config.yaml").exists() else {}
        m = json.loads((d / "metrics.json").read_text())
        summary.loc[d.name, "family"] = c.get("family", summary.family.get(d.name, ""))
        summary.loc[d.name, "test-like val mAP"] = like(pd.read_csv(d / "predictions" / "val.csv"))["mAP"]
        if "val" in m:
            summary.loc[d.name, ["full val mAP", "AP_small", "ms_per_img"]] = [m["val"]["mAP"], m["val"]["AP_small"], m["val"]["ms_per_img"]]
        else:
            summary.loc[d.name, "ms_per_img"] = m["val_like_test"]["ms_per_img"]
summary = summary.sort_values(["test-like val mAP", "full val mAP"], ascending=False)
summary.round(3)

In [ ]:
# Bar chart: test-like val mAP of every run with saved val predictions, coloured by family
families = ["two_stage", "one_stage", "anchor_free", "query_based", "pipeline"]
s = summary.dropna(subset=["test-like val mAP"]).sort_values("test-like val mAP")
fig, ax = plt.subplots(figsize=(8, 0.32 * len(s) + 1))
ax.barh(s.index, s["test-like val mAP"], color=[PALETTE[families.index(f)] if f in families else INK_3 for f in s.family], height=0.7)
for fam in families:
    if fam in set(s.family):
        ax.barh([], [], color=PALETTE[families.index(fam)], label=fam.replace("_", "-"))
for y, v in enumerate(s["test-like val mAP"]):
    ax.text(v + 0.002, y, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("mAP@[.5:.95], val frames at most 1920 px wide")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
save_fig(fig, "pipeline2_summary", P)
plt.show()

In [ ]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 3600:.2f} h")
env.warnings_summary(P)